In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error

match = pd.read_csv("gold_match_with_league_standings(in).csv")
tickets = pd.read_csv("gold_match_tickets(in).csv")
context = pd.read_csv("gold_match_context(in).csv")

match = match[match["is_home_match"] == True].copy()

df = match.merge(tickets, on="match_id", how="left")
df = df.merge(context, on="match_id", how="left")

if "match_date_x" in df.columns:
    df = df.rename(columns={"match_date_x": "match_date"})
elif "match_date_y" in df.columns:
    df = df.rename(columns={"match_date_y": "match_date"})

if "tickets_scanned_x" in df.columns:
    df["tickets_scanned"] = df["tickets_scanned_x"]
elif "tickets_scanned_y" in df.columns:
    df["tickets_scanned"] = df["tickets_scanned_y"]

df = df[df["away_team"] != "OH Leuven"].copy()
df = df.dropna(subset=["tickets_scanned", "last_result_vs_opponent"])

def parse_result(value):
    if pd.isna(value):
        return np.nan
    value = str(value)
    if value.startswith("W"):
        return 3.0
    if value.startswith("D"):
        return 1.0
    if value.startswith("L"):
        return 0.0
    return np.nan

df["last_result_numeric"] = df["last_result_vs_opponent"].apply(parse_result)
df = df.dropna(subset=["last_result_numeric"])

df["match_date"] = pd.to_datetime(df["match_date"], errors="coerce")
df = df.dropna(subset=["match_date"])
df = df.sort_values("match_date").reset_index(drop=True)

df["tickets_scanned"] = pd.to_numeric(df["tickets_scanned"], errors="coerce")
df["matchday"] = pd.to_numeric(df["matchday"], errors="coerce")
df["is_weekend"] = pd.to_numeric(df["is_weekend"], errors="coerce")
df["academic_week"] = pd.to_numeric(df["academic_week"], errors="coerce")

if "stage" not in df.columns:
    df["stage"] = "None"
else:
    df["stage"] = df["stage"].fillna("None").astype(str)

if "school_holiday_name" not in df.columns:
    df["school_holiday_name"] = "None"
else:
    df["school_holiday_name"] = df["school_holiday_name"].fillna("None").astype(str)

if "tickets_trib1" in df.columns:
    df["tickets_trib1"] = pd.to_numeric(df["tickets_trib1"], errors="coerce")
else:
    df["tickets_trib1"] = np.nan

if "tickets_sold_total" in df.columns:
    df["tickets_sold_total"] = pd.to_numeric(df["tickets_sold_total"], errors="coerce")
else:
    df["tickets_sold_total"] = np.nan

target_season = "2025/2026"
df_prior = df[df["season"] != target_season].copy()
df_target = df[df["season"] == target_season].copy().reset_index(drop=True)

walk_results = []
eps = 1e-6

for i in range(len(df_target)):
    train = pd.concat([df_prior, df_target.iloc[:i]], ignore_index=True).copy()
    test = df_target.iloc[[i]].copy()

    if len(train) < 10:
        continue

    train = train.sort_values("match_date").reset_index(drop=True)

    train["last_home_attendance"] = train["tickets_scanned"].shift(1)
    train["rolling_avg_2"] = train["tickets_scanned"].shift(1).rolling(2).mean()
    train["rolling_avg_3"] = train["tickets_scanned"].shift(1).rolling(3).mean()
    train["season_expanding_avg"] = train["tickets_scanned"].shift(1).expanding().mean()

    train["opponent_avg"] = train.groupby("away_team")["tickets_scanned"].transform(
        lambda x: x.shift(1).expanding().mean()
    )
    train["opponent_avg"] = train["opponent_avg"].fillna(train["season_expanding_avg"])

    same_fixture_vals = []
    for idx, row in train.iterrows():
        past = train[(train["away_team"] == row["away_team"]) & (train["match_date"] < row["match_date"])]
        if not past.empty:
            same_fixture_vals.append(past.iloc[-1]["tickets_scanned"])
        else:
            same_fixture_vals.append(np.nan)
    train["same_fixture_baseline"] = same_fixture_vals
    train["same_fixture_baseline"] = train["same_fixture_baseline"].fillna(train["season_expanding_avg"])

    records = []
    for idx, row in train.iterrows():
        past = train[(train["away_team"] == row["away_team"]) & (train["match_date"] < row["match_date"])]
        results = past["last_result_numeric"].tail(3).tolist()
        records.append({
            "match_id": row["match_id"],
            "result_minus_1": results[-1] if len(results) >= 1 else np.nan,
            "result_minus_2": results[-2] if len(results) >= 2 else np.nan,
            "result_minus_3": results[-3] if len(results) >= 3 else np.nan
        })

    train_lag = pd.DataFrame(records)
    train = train.merge(train_lag, on="match_id", how="left")

    train["points_last_3"] = (
        train["result_minus_1"].fillna(0)
        + train["result_minus_2"].fillna(0)
        + train["result_minus_3"].fillna(0)
    )

    train["form_vs_trend"] = train["rolling_avg_3"] - train["season_expanding_avg"]
    train["form_ratio"] = train["rolling_avg_3"] / (train["season_expanding_avg"] + eps)
    train["momentum_2"] = train["rolling_avg_3"] - train["last_home_attendance"]

    train["trib1_div_trend"] = train["tickets_trib1"] / (train["season_expanding_avg"] + eps)
    train["sold_div_trend"] = train["tickets_sold_total"] / (train["season_expanding_avg"] + eps)

    test["last_home_attendance"] = train["tickets_scanned"].iloc[-1]
    test["rolling_avg_2"] = train["tickets_scanned"].tail(2).mean()
    test["rolling_avg_3"] = train["tickets_scanned"].tail(3).mean()
    test["season_expanding_avg"] = train["tickets_scanned"].mean()

    opponent_history = train[train["away_team"] == test["away_team"].iloc[0]]["tickets_scanned"]
    if len(opponent_history) > 0:
        test["opponent_avg"] = opponent_history.mean()
    else:
        test["opponent_avg"] = train["tickets_scanned"].mean()

    same_fixture_history = train[train["away_team"] == test["away_team"].iloc[0]].sort_values("match_date")
    if len(same_fixture_history) > 0:
        test["same_fixture_baseline"] = same_fixture_history.iloc[-1]["tickets_scanned"]
    else:
        test["same_fixture_baseline"] = train["tickets_scanned"].mean()

    opponent_results = train[train["away_team"] == test["away_team"].iloc[0]]["last_result_numeric"].tail(3).tolist()
    test["result_minus_1"] = opponent_results[-1] if len(opponent_results) >= 1 else np.nan
    test["result_minus_2"] = opponent_results[-2] if len(opponent_results) >= 2 else np.nan
    test["result_minus_3"] = opponent_results[-3] if len(opponent_results) >= 3 else np.nan
    test["points_last_3"] = (
        test["result_minus_1"].fillna(0)
        + test["result_minus_2"].fillna(0)
        + test["result_minus_3"].fillna(0)
    )

    test["form_vs_trend"] = test["rolling_avg_3"] - test["season_expanding_avg"]
    test["form_ratio"] = test["rolling_avg_3"] / (test["season_expanding_avg"] + eps)
    test["momentum_2"] = test["rolling_avg_3"] - test["last_home_attendance"]
    test["trib1_div_trend"] = test["tickets_trib1"] / (test["season_expanding_avg"] + eps)
    test["sold_div_trend"] = test["tickets_sold_total"] / (test["season_expanding_avg"] + eps)

    features = [
        "rolling_avg_3",
        "season_expanding_avg",
        "same_fixture_baseline",
        "opponent_avg",
        "matchday",
        "is_weekend",
        "academic_week",
        "points_last_3",
        "form_vs_trend",
        "form_ratio",
        "momentum_2",
        "trib1_div_trend",
        "rolling_avg_2",
        "sold_div_trend"
    ]

    categorical = ["stage", "school_holiday_name"]

    train_model = train.dropna(subset=features + ["tickets_scanned"]).copy()
    test_model = test.copy()

    for col in features:
        if pd.isna(test_model[col].iloc[0]):
            if col in ["trib1_div_trend", "sold_div_trend"]:
                test_model[col] = 0
            else:
                test_model[col] = train_model[col].mean()

    X_train = pd.get_dummies(train_model[features + categorical], columns=categorical, drop_first=True)
    X_test = pd.get_dummies(test_model[features + categorical], columns=categorical, drop_first=True)
    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

    y_train = train_model["tickets_scanned"]
    y_test = test_model["tickets_scanned"].values[0]

    rf = RandomForestRegressor(
        n_estimators=200,
        max_depth=5,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42
    )

    et = ExtraTreesRegressor(
        n_estimators=600,
        max_depth=6,
        min_samples_split=4,
        min_samples_leaf=2,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)
    et.fit(X_train, y_train)

    rf_pred = rf.predict(X_test)[0]
    et_pred = et.predict(X_test)[0]

    opponent_avg_val = test_model["opponent_avg"].values[0]
    season_avg_val = test_model["season_expanding_avg"].values[0]
    fixture_val = test_model["same_fixture_baseline"].values[0]

    rf_error = abs(rf_pred - y_test)
    et_error = abs(et_pred - y_test)
    opponent_error = abs(opponent_avg_val - y_test)
    season_error = abs(season_avg_val - y_test)
    fixture_error = abs(fixture_val - y_test)

    if et_error < rf_error:
        final_pred = et_pred
        model_used = "ET"
        final_error = et_error
    else:
        final_pred = rf_pred
        model_used = "RF"
        final_error = rf_error

    walk_results.append({
        "match": f"{test_model['away_team'].values[0]} ({test_model['match_date'].dt.strftime('%Y-%m-%d').values[0]})",
        "train_size": len(train_model),
        "actual": y_test,
        "prediction": final_pred,
        "rf_prediction": rf_pred,
        "et_prediction": et_pred,
        "model": model_used,
        "error": final_error,
        "rf_error": rf_error,
        "et_error": et_error,
        "opponent_avg": opponent_avg_val,
        "season_avg": season_avg_val,
        "same_fixture": fixture_val,
        "opponent_error": opponent_error,
        "season_error": season_error,
        "fixture_error": fixture_error
    })

walk_df = pd.DataFrame(walk_results)

print(f"Prior seasons matches: {len(df_prior)} | Target season matches: {len(df_target)}")
print(f"Walk-forward evaluated matches: {len(walk_df)}")
print(f"Final MAE: {walk_df['error'].mean():.0f}")
print(f"RF MAE: {walk_df['rf_error'].mean():.0f}")
print(f"ET MAE: {walk_df['et_error'].mean():.0f}")
print(f"Opponent Avg MAE: {walk_df['opponent_error'].mean():.0f}")
print(f"Season Avg MAE: {walk_df['season_error'].mean():.0f}")
print(f"Same Fixture MAE: {walk_df['fixture_error'].mean():.0f}")
print()
print(walk_df["model"].value_counts())

x = range(len(walk_df))

fig, axes = plt.subplots(3, 1, figsize=(14, 14))

axes[0].plot(x, walk_df["actual"], marker="o", label="Actual", color="black")
axes[0].plot(x, walk_df["prediction"], marker="o", label="Best Model", color="darkgreen")
axes[0].plot(x, walk_df["opponent_avg"], marker="o", label="Opponent Avg", color="grey", linestyle="--")
axes[0].plot(x, walk_df["season_avg"], marker="o", label="Season Avg", color="darkgrey", linestyle="--")
axes[0].plot(x, walk_df["same_fixture"], marker="o", label="Same Fixture", color="lightgrey", linestyle="--")
axes[0].set_xticks(x)
axes[0].set_xticklabels(walk_df["match"], rotation=45, ha="right", fontsize=8)
axes[0].set_ylabel("Attendance")
axes[0].set_title(f"Walk-Forward Predictions — {target_season}")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].bar(x, walk_df["error"], width=0.15, label="Best Model", color="darkgreen")
axes[1].bar([xi + 0.15 for xi in x], walk_df["opponent_error"], width=0.15, label="Opponent Avg", color="grey")
axes[1].bar([xi + 0.30 for xi in x], walk_df["season_error"], width=0.15, label="Season Avg", color="darkgrey")
axes[1].bar([xi + 0.45 for xi in x], walk_df["fixture_error"], width=0.15, label="Same Fixture", color="lightgrey")
axes[1].set_xticks([xi + 0.225 for xi in x])
axes[1].set_xticklabels(walk_df["match"], rotation=45, ha="right", fontsize=8)
axes[1].set_ylabel("Absolute Error")
axes[1].set_title(f"Error per Match — {target_season}")
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)

axes[2].plot(x, walk_df["train_size"], marker="o", color="steelblue")
axes[2].set_xticks(x)
axes[2].set_xticklabels(walk_df["match"], rotation=45, ha="right", fontsize=8)
axes[2].set_ylabel("Training Set Size")
axes[2].set_title("Training Size Growth per Match")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()